# Assignment 4: Naive Bayes Classification
## Part 1: Gaussian Naive Bayes (Bank Churn Dataset)
## Part 2: Multinomial Naive Bayes (SMS Spam Dataset)

Included:

- Confusion Matrix
- Classification Report
- Top Spam Words Analysis


## ------------------ PART 1: Gaussian Naive Bayes ------------------

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load Churn Dataset
df = pd.read_csv('Churn_Modelling.csv')

df = df.drop(['RowNumber','CustomerId','Surname'], axis=1)
df = pd.get_dummies(df, drop_first=True)

X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred = gnb.predict(X_test)

print('Gaussian NB Accuracy:', accuracy_score(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

Gaussian NB Accuracy: 0.8335

Confusion Matrix:
 [[1526   81]
 [ 252  141]]

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.95      0.90      1607
           1       0.64      0.36      0.46       393

    accuracy                           0.83      2000
   macro avg       0.75      0.65      0.68      2000
weighted avg       0.81      0.83      0.81      2000



## ------------------ PART 2: Multinomial Naive Bayes ------------------

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

# Load SMS Spam Dataset
# The file has literal \t delimiters, so we need to split manually
data = []
with open('SMSSpamCollection.csv', 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t', 1)  # Split on first tab only
        if len(parts) == 2:
            data.append({'label': parts[0].strip('"'), 'message': parts[1]})  # Remove quotes from label

sms = pd.DataFrame(data)
print(f"Total rows loaded: {len(sms)}")
print(f"Unique labels: {sms['label'].unique()}")

# Map labels to numeric values
sms['label'] = sms['label'].map({'ham':0, 'spam':1})

# Remove any rows where label mapping failed (resulted in NaN)
sms = sms.dropna(subset=['label'])

print(f"Rows after cleaning: {len(sms)}")

X_text = sms['message']
y_text = sms['label']

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_text, y_text, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train_t)
X_test_vec = vectorizer.transform(X_test_t)

mnb = MultinomialNB()
mnb.fit(X_train_vec, y_train_t)

y_pred_text = mnb.predict(X_test_vec)

print('Multinomial NB Accuracy:', accuracy_score(y_test_t, y_pred_text))
print('\nConfusion Matrix:\n', confusion_matrix(y_test_t, y_pred_text))
print('\nClassification Report:\n', classification_report(y_test_t, y_pred_text))

Total rows loaded: 5574
Unique labels: ['ham' 'spam']
Rows after cleaning: 5574
Multinomial NB Accuracy: 0.9739910313901345

Confusion Matrix:
 [[954   0]
 [ 29 132]]

Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99       954
           1       1.00      0.82      0.90       161

    accuracy                           0.97      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [8]:
# Top Spam Words (Innovation)
feature_names = np.array(vectorizer.get_feature_names_out())
spam_probs = mnb.feature_log_prob_[1]
top_spam = feature_names[np.argsort(spam_probs)[-10:]]

print('Top words indicating Spam:')
print(top_spam)

Top words indicating Spam:
['prize' 'www' 'ur' 'reply' 'text' 'stop' 'claim' 'mobile' 'txt' 'free']
